# Phase 08 — Unsupervised Network Anomaly Detection

## 1. Project introduction
This notebook reviews the reusable benign-baseline anomaly workflow for CIC-IDS2017. The verified run is 20261103T141439Z. It loads the recorded run by default and does not silently retrain or overwrite versioned artifacts.

## 2. Anomaly detection methodology
Isolation Forest learns from benign training flows only. A separate benign validation population sets the operating threshold. Labels are retained for final held-out evaluation only. An anomaly score means unusual relative to the observed baseline, not confirmed malicious traffic or a probability of compromise.


## 3. Dataset loading
## 4. Benign label identification
## 5. Original attack-category inspection
## 6. Predictive feature and metadata separation
## 7. Group-aware data splitting

The source modules validate Phase 04 Parquet files against the Phase 05 feature contract, retain exact original labels, and assign every capture day exactly once.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from intrusion_intelligence.anomaly.config import load_anomaly_config
from intrusion_intelligence.anomaly.data import inspect_anomaly_dataset, prepare_anomaly_frame
from intrusion_intelligence.anomaly.splitting import make_capture_day_split
from intrusion_intelligence.feature_engineering import FeatureContract

CONFIG_PATH = Path("configs/anomaly_detection.toml")
config = load_anomaly_config(CONFIG_PATH)
contract = FeatureContract.from_file(config["dataset"]["feature_schema"])
inventory = inspect_anomaly_dataset(
    config["dataset"]["parquet_directory"], contract,
    benign_label=config["dataset"]["benign_label"],
)
split = make_capture_day_split(inventory, config, benign_label=config["dataset"]["benign_label"])

print(inventory.preparation_report())
print("Partition rows:", split.audit["partition_rows"])
print("Partition BENIGN rows:", split.audit["partition_benign_rows"])
print("Partition attack rows:", split.audit["partition_attack_rows"])
print("Capture-day groups disjoint:", split.audit["capture_day_groups_disjoint"])
print("Source-file groups disjoint:", split.audit["source_files_disjoint"])
display(pd.DataFrame(split.audit["class_counts"]).fillna(0).astype(int))


In [ ]:
# Read only a small batch to demonstrate Phase 05 X/y/metadata separation.
monday_path = next(path for path in inventory.files if path.name.startswith("Monday"))
batch_columns = list(contract.predictive_features) + [contract.target_name, "__source_file"]
sample_batch = next(pq.ParquetFile(monday_path).iter_batches(batch_size=128, columns=batch_columns))
sample_frame = sample_batch.to_pandas()
X_sample, y_sample, metadata_sample, benign_mask = prepare_anomaly_frame(sample_frame, contract)
assert tuple(X_sample.columns) == tuple(contract.predictive_features)
assert contract.target_name not in X_sample.columns
assert "__source_file" not in X_sample.columns
print("Feature shape:", X_sample.shape, "| metadata fields:", list(metadata_sample.columns))
print("Original labels:", y_sample.value_counts().to_dict(), "| benign rows:", int(benign_mask.sum()))


## 8. Benign-only training dataset preparation
## 9. Benign validation dataset preparation
## 10. Held-out test dataset preparation

Training uses a deterministic 150,000-row BENIGN sample from Monday–Wednesday. Thursday BENIGN rows calibrate thresholds. Friday is held out for labelled evaluation; Thursday attacks do not enter calibration.

## 11. Anomaly preprocessing configuration
## 12. Training-only preprocessing fitting

The independent Phase 05 anomaly profile uses median imputation and RobustScaler. The source service fits it on the benign training sample and reuses it unchanged for validation, test, and inference.

## 13. Isolation Forest training

The real model has 200 estimators, max_samples=4096, max_features=1.0, contamination='auto', no bootstrap, seed 42, and four jobs. Training and persistence are implemented in the reusable service.


In [ ]:
RUN_ID = "20261103T141439Z"
RUN_DIR = Path("reports/experiments/anomaly_detection") / RUN_ID
EVALUATION_DIR = Path("reports/evaluation/anomaly_detection") / RUN_ID
summary = json.loads((RUN_DIR / "experiment_summary.json").read_text(encoding="utf-8"))
calibration = json.loads((EVALUATION_DIR / "threshold_calibration.json").read_text(encoding="utf-8"))

display(pd.Series({
    "dataset_rows": summary["dataset_rows"],
    "features": summary["feature_count"],
    "benign_train_population": summary["benign_training_rows"],
    "benign_fit_sample": summary["training_sample_rows"],
    "benign_validation_population": summary["benign_validation_rows"],
    "test_rows": summary["test_rows"],
    "preprocessing_profile": summary["preprocessing_profile"],
    "model_configuration": summary["model_configuration"],
    "training_seconds": summary["training_time_seconds"],
    "test_prediction_seconds": summary["test_prediction_time_seconds"],
}).to_frame("actual_run"))

# The saved 1.0.0 versions are immutable. Use unused versions before retraining.
RUN_NEW_EXPERIMENT = False
if RUN_NEW_EXPERIMENT:
    from intrusion_intelligence.anomaly.service import run_anomaly_experiment
    new_summary = run_anomaly_experiment(CONFIG_PATH)
    display(new_summary)
else:
    print("Loaded the recorded trained model; no fitting or artifact writes ran in this notebook.")


## 14. Anomaly-score generation
## 15. Score direction validation

Isolation Forest score_samples is inlier-oriented. The public convention negates it, so a higher common anomaly score means more unusual. Scores are not probabilities and are not comparable across different anomaly algorithms.

## 16. Threshold calibration

Thresholds are empirical upper quantiles of benign Thursday scores, using the higher quantile method. The decision rule is score > threshold; threshold ties are NORMAL. The chosen 1% operating point is configured independently of test labels.


In [ ]:
operating_points = calibration["operating_points"]
threshold_table = pd.DataFrame.from_dict(operating_points, orient="index")[[
    "target_fpr", "threshold", "achieved_validation_fpr",
    "benign_validation_observations", "threshold_tie_observations", "empirical_fpr_resolution",
]]
display(threshold_table)
assert calibration["labels_used_for_threshold_selection"] is False
print("Default point:", summary["default_operating_point"])
print("Calibration population:", calibration["calibration_population"])


## 17. Benign validation false-positive analysis

The calibration file records the actual empirical validation FPR and finite-sample resolution for each operating point. These values describe Thursday BENIGN traffic only.

## 18. Held-out test evaluation

Test labels are used only after fitting and threshold calibration are fixed. The 1% validation-FPR threshold does not guarantee a 1% Friday FPR.

## 19. Per-attack-category detection analysis

Only attack labels represented in Friday have an evaluable recall. Missing categories are marked unavailable.


In [ ]:
default_point = summary["default_operating_point"]
test_metric = summary["test_metrics"][default_point]
display(pd.Series(test_metric).to_frame("held_out_test"))
attack_table = pd.read_csv(EVALUATION_DIR / "per_attack_category.csv")
display(attack_table)


## 20. False-positive analysis

False positives are grouped by Friday source file to identify capture-level shifts. The report avoids exposing raw record identifiers.

## 21. False-negative analysis

False negatives are reported by original attack category and supported by bounded, deterministic feature-median summaries. A missed attack resembles the observed benign baseline under this model; it is not re-labeled as benign.

## 22. Threshold sensitivity

Each point uses a separately calibrated benign-validation threshold. Test curves are descriptive evaluation and do not select a threshold.


In [ ]:
false_positive_table = pd.read_csv(EVALUATION_DIR / "false_positive_by_source_file.csv")
false_negative_table = pd.read_csv(EVALUATION_DIR / "false_negative_by_attack_category.csv")
sensitivity_table = pd.read_csv(EVALUATION_DIR / "threshold_sensitivity.csv")
display(false_positive_table)
display(false_negative_table[false_negative_table["test_support"] > 0])
display(sensitivity_table)


## 23. Alternative model comparison where implemented

Only Isolation Forest was trained. One-Class SVM and Local Outlier Factor were investigated but not implemented because kernel training and neighbor-index storage/scoring are costly at this scale.

## 24. Model persistence

The model and fitted anomaly preprocessor are stored as a versioned compatible bundle. Existing versions are not overwritten.

## 25. Prediction interface demonstration

The predictor receives feature columns only, loads frozen artifacts, applies an existing operating point, and does not refit or recalibrate.


In [ ]:
from intrusion_intelligence.anomaly.inference import AnomalyPredictor

model_directory = config["artifacts"]["model_directory"] / config["artifacts"]["model_version"]
predictor = AnomalyPredictor(model_directory)
comparison = pd.read_csv(EVALUATION_DIR / "model_comparison.csv")
display(comparison)

friday_path = next(path for path in inventory.files if path.name.startswith("Friday"))
inference_batch = next(pq.ParquetFile(friday_path).iter_batches(
    batch_size=8, columns=list(contract.predictive_features)
))
X_inference = inference_batch.to_pandas()
predictions = predictor.predict(X_inference, operating_point=default_point)
assert np.allclose(predictions["anomaly_score"], -predictions["raw_model_score"])
display(predictions)
print("Frozen model:", predictor.metadata["model_id"])


## 26. Findings and limitations

At the 1% benign-validation operating point, Friday test FPR was 4.91%, attack recall 15.57%, and ROC-AUC 0.6941. DDoS recall was 35.06%, PortScan recall 0.028%, and Bot recall 2.24%. The benign DDoS-capture file had a 16.69% false-positive rate, indicating a substantial source/capture shift.

Repeated feature vectors cross all five capture days: 28,644 groups cross selected partitions, including 633 groups with conflicting labels. Capture-day groups are disjoint, but the benchmark is not duplicate-disjoint. Only three attack categories appear in Friday test data. The 150,000-row training sample is smaller than the available benign training population. Phase 06 binary classification remains not started; no supervised/anomaly combination, threat scoring, or Phase 09 alert integration is implemented. See docs/anomaly_detection.md for full results and artifact paths.
